**Preprocessing strategies**

In [ ]:
1. Mount Google Drive and imports

In [ ]:
# ============================================================
#  CORE LIBRARIES FOR DATA AND IMAGE PROCESSING PIPELINE
# ============================================================

# File system and directory management
import os
import shutil
import random

# Numerical computing
import numpy as np

# Image processing and computer vision
import cv2

# Progress bar utilities
from tqdm import tqdm

# Google Drive integration (Colab environment)
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


2. DATASET_liver_stu_LAB (ABL-1)

In [ ]:
# =========================
# CONFIG
# =========================
INPUT_DIR  = "/content/drive/MyDrive/Challenge_Liver_MIP/DATASET_liver_stu"
OUTPUT_DIR = "/content/drive/MyDrive/Challenge_Liver_MIP/DATASET_liver_stu_LAB"

# Subfolders to process (train / val / test)
SPLITS = ["train", "val", "test"]
IMG_SUBDIR = "image"

# LAB-safe parameters
L_TARGET_MEAN = 128.0
L_TARGET_STD  = 40.0
L_CLIP_MIN    = 20
L_CLIP_MAX    = 235

# =========================
# UTILS
# =========================
def tissue_mask_rgb(img_rgb):
    """
    Simple tissue mask:
    - background is very bright and low saturation
    """
    hsv = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2HSV)
    s = hsv[:, :, 1]
    v = hsv[:, :, 2]

    # tissue: not too bright, some saturation
    mask = (v < 240) & (s > 15)
    return mask.astype(np.uint8)

def lab_safe_normalize(img_rgb):
    lab = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2LAB)
    L, A, B = cv2.split(lab)

    mask = tissue_mask_rgb(img_rgb)

    if mask.sum() < 50:
        # almost no tissue → return original
        return img_rgb

    L_tissue = L[mask == 1].astype(np.float32)

    mean = L_tissue.mean()
    std  = L_tissue.std() + 1e-6

    # normalize L only
    L_norm = (L.astype(np.float32) - mean) / std
    L_norm = L_norm * L_TARGET_STD + L_TARGET_MEAN
    L_norm = np.clip(L_norm, L_CLIP_MIN, L_CLIP_MAX).astype(np.uint8)

    lab_norm = cv2.merge([L_norm, A, B])
    rgb_norm = cv2.cvtColor(lab_norm, cv2.COLOR_LAB2RGB)

    return rgb_norm

# =========================
# MAIN LOOP
# =========================
for split in SPLITS:
    in_dir  = os.path.join(INPUT_DIR, split, IMG_SUBDIR)
    out_dir = os.path.join(OUTPUT_DIR, split, IMG_SUBDIR)
    os.makedirs(out_dir, exist_ok=True)

    if not os.path.exists(in_dir):
        continue

    files = [f for f in os.listdir(in_dir) if f.lower().endswith((".png", ".jpg", ".jpeg", ".tif"))]

    print(f"\nProcessing {split}: {len(files)} images")

    for fname in tqdm(files, ncols=80):
        img_path = os.path.join(in_dir, fname)
        out_path = os.path.join(out_dir, fname)

        img_bgr = cv2.imread(img_path)
        if img_bgr is None:
            continue

        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        img_lab_safe = lab_safe_normalize(img_rgb)

        cv2.imwrite(out_path, cv2.cvtColor(img_lab_safe, cv2.COLOR_RGB2BGR))

print("\n✅ LAB-safe normalization completed.")
print("Output dataset:", OUTPUT_DIR)


3. DATASET_liver_stu_LAB_ABL_2

In [ ]:
# =========================
# CONFIG
# =========================
INPUT_DIR  = "/content/drive/MyDrive/Challenge_Liver_MIP/DATASET_liver_stu"
OUTPUT_DIR = "/content/drive/MyDrive/Challenge_Liver_MIP/DATASET_liver_stu_LAB_ABL_2"

# Subfolders to process (train / val / test)
SPLITS = ["train", "val", "test"]
IMG_SUBDIR = "image"

# LAB-safe parameters
L_TARGET_MEAN = 145.0
L_TARGET_STD = 25.0
L_CLIP_MIN = 10
L_CLIP_MAX = 245

# =========================
# UTILS
# =========================
def tissue_mask_rgb(img_rgb):
    """
    Simple tissue mask:
    - background is very bright and low saturation
    """
    hsv = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2HSV)
    s = hsv[:, :, 1]
    v = hsv[:, :, 2]

    # tissue: not too bright, some saturation
    mask = (v < 245) & (s > 10)
    return mask.astype(np.uint8)

def lab_safe_normalize(img_rgb):
    lab = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2LAB)
    L, A, B = cv2.split(lab)

    mask = tissue_mask_rgb(img_rgb)

    if mask.sum() < 50:
        # almost no tissue → return original
        return img_rgb

    L_tissue = L[mask == 1].astype(np.float32)

    mean = L_tissue.mean()
    std  = L_tissue.std() + 1e-6

    # normalize L only
    L_norm = (L.astype(np.float32) - mean) / std
    L_norm = L_norm * L_TARGET_STD + L_TARGET_MEAN
    L_norm = np.clip(L_norm, L_CLIP_MIN, L_CLIP_MAX).astype(np.uint8)

    lab_norm = cv2.merge([L_norm, A, B])
    rgb_norm = cv2.cvtColor(lab_norm, cv2.COLOR_LAB2RGB)

    return rgb_norm

# =========================
# MAIN LOOP
# =========================
for split in SPLITS:
    in_dir  = os.path.join(INPUT_DIR, split, IMG_SUBDIR)
    out_dir = os.path.join(OUTPUT_DIR, split, IMG_SUBDIR)
    os.makedirs(out_dir, exist_ok=True)

    if not os.path.exists(in_dir):
        continue

    files = [f for f in os.listdir(in_dir) if f.lower().endswith((".png", ".jpg", ".jpeg", ".tif"))]

    print(f"\nProcessing {split}: {len(files)} images")

    for fname in tqdm(files, ncols=80):
        img_path = os.path.join(in_dir, fname)
        out_path = os.path.join(out_dir, fname)

        img_bgr = cv2.imread(img_path)
        if img_bgr is None:
            continue

        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        img_lab_safe = lab_safe_normalize(img_rgb)

        cv2.imwrite(out_path, cv2.cvtColor(img_lab_safe, cv2.COLOR_RGB2BGR))

print("\n✅ LAB-safe normalization completed.")
print("Output dataset:", OUTPUT_DIR)

4. DATASET_liver_stu_CLAHE_L (ABL-3)

In [ ]:
# =========================
# CONFIG
# =========================
INPUT_DIR  = "/content/drive/MyDrive/Challenge_Liver_MIP/DATASET_liver_stu"
OUTPUT_DIR = "/content/drive/MyDrive/Challenge_Liver_MIP/DATASET_liver_stu_CLAHE_L"

SPLITS = ["train", "val", "test"]
IMG_SUBDIRS = ["image"]
MASK_SUBDIRS_TO_COPY = ["manual", "manual_py"]

# CLAHE params (safe defaults)
CLIP_LIMIT = 2.0
TILE_GRID  = (8, 8)
clahe = cv2.createCLAHE(clipLimit=CLIP_LIMIT, tileGridSize=TILE_GRID)

# =========================
# HELPERS
# =========================
def is_image_file(fname):
    f = fname.lower()
    return f.endswith((".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp"))

def process_image_clahe_L(in_path, out_path):
    bgr = cv2.imread(in_path)
    if bgr is None:
        return False

    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    lab = cv2.cvtColor(rgb, cv2.COLOR_RGB2LAB)
    L, A, B = cv2.split(lab)

    L2 = clahe.apply(L)

    lab2 = cv2.merge([L2, A, B])
    rgb2 = cv2.cvtColor(lab2, cv2.COLOR_LAB2RGB)
    bgr2 = cv2.cvtColor(rgb2, cv2.COLOR_RGB2BGR)

    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    cv2.imwrite(out_path, bgr2)
    return True

def copy_folder_if_exists(src, dst):
    if os.path.exists(src):
        os.makedirs(dst, exist_ok=True)
        for fname in os.listdir(src):
            in_path = os.path.join(src, fname)
            out_path = os.path.join(dst, fname)
            if os.path.isfile(in_path):
                shutil.copy2(in_path, out_path)

# =========================
# MAIN
# =========================
for split in SPLITS:
    print(f"\n=== Split: {split} ===")

    # 1) Process images
    for sub in IMG_SUBDIRS:
        in_dir  = os.path.join(INPUT_DIR, split, sub)
        out_dir = os.path.join(OUTPUT_DIR, split, sub)
        os.makedirs(out_dir, exist_ok=True)

        if not os.path.exists(in_dir):
            print(f"⚠️ Missing folder (skip): {in_dir}")
            continue

        files = [f for f in os.listdir(in_dir) if is_image_file(f)]
        print(f"Processing {split}/{sub}: {len(files)} images")

        for fname in tqdm(files, ncols=80):
            ok = process_image_clahe_L(
                in_path=os.path.join(in_dir, fname),
                out_path=os.path.join(out_dir, fname)
            )
            if not ok:
                print("⚠️ Could not read:", os.path.join(in_dir, fname))

    # 2) Copy masks folders as-is
    for msub in MASK_SUBDIRS_TO_COPY:
        src = os.path.join(INPUT_DIR, split, msub)
        dst = os.path.join(OUTPUT_DIR, split, msub)
        if os.path.exists(src):
            print(f"Copying {split}/{msub} ...")
            copy_folder_if_exists(src, dst)
        else:
            print(f"⚠️ Missing folder (skip): {src}")

print("\n✅ DONE. New dataset created at:")
print(OUTPUT_DIR)

5. DATASET_liver_stu_REINHARD (ABL-4)

In [ ]:
# =========================
# CONFIG
# =========================
INPUT_DIR  = "/content/drive/MyDrive/Challenge_Liver_MIP/DATASET_liver_stu"
OUTPUT_DIR = "/content/drive/MyDrive/Challenge_Liver_MIP/DATASET_liver_stu_REINHARD"

SPLITS = ["train", "val", "test"]
IMG_SUBDIR = "image"
MASK_SUBDIRS_TO_COPY = ["manual", "manual_py"]

# Pick ONE representative training image as target (tissue-rich, typical stain)
TARGET_IMG_PATH = "/content/drive/MyDrive/Challenge_Liver_MIP/DATASET_liver_stu/train/image/2016T0249_7.png"

# Outlier clipping for LAB channels (robustness)
CLIP_PCT_LOW  = 1.0
CLIP_PCT_HIGH = 99.0

# =========================
# UTILS
# =========================
def is_image_file(fname):
    f = fname.lower()
    return f.endswith((".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp"))

def tissue_mask_otsu_L(img_rgb):
    lab = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2LAB)
    L = lab[:, :, 0]
    # tissue darker than background -> threshold on inverted L
    inv = 255 - L
    _, th = cv2.threshold(inv, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    mask = (th > 0).astype(np.uint8)

    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, k, iterations=1)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, k, iterations=2)
    return mask

def robust_mean_std(channel, mask):
    vals = channel[mask == 1].astype(np.float32)
    if vals.size < 50:
        return None, None

    lo = np.percentile(vals, CLIP_PCT_LOW)
    hi = np.percentile(vals, CLIP_PCT_HIGH)
    vals = np.clip(vals, lo, hi)

    return float(vals.mean()), float(vals.std() + 1e-6)

def reinhard_normalize(img_rgb, target_stats):
    lab = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2LAB).astype(np.float32)
    L, A, B = lab[:, :, 0], lab[:, :, 1], lab[:, :, 2]

    mask = tissue_mask_otsu_L(img_rgb)
    if mask.sum() < 50:
        return img_rgb  # too little tissue

    src_stats = []
    for ch in [L, A, B]:
        m, s = robust_mean_std(ch, mask)
        if m is None:
            return img_rgb
        src_stats.append((m, s))

    # Apply per-channel normalization only on tissue pixels
    out = lab.copy()
    for i, ch in enumerate([L, A, B]):
        src_m, src_s = src_stats[i]
        tgt_m, tgt_s = target_stats[i]
        norm = (ch - src_m) / src_s
        norm = norm * tgt_s + tgt_m
        out[:, :, i] = norm

    out = np.clip(out, 0, 255).astype(np.uint8)
    rgb_out = cv2.cvtColor(out, cv2.COLOR_LAB2RGB)
    return rgb_out

def copy_folder_if_exists(src, dst):
    if os.path.exists(src):
        os.makedirs(dst, exist_ok=True)
        for fname in os.listdir(src):
            in_path = os.path.join(src, fname)
            out_path = os.path.join(dst, fname)
            if os.path.isfile(in_path):
                shutil.copy2(in_path, out_path)

# =========================
# TARGET STATS
# =========================
assert os.path.exists(TARGET_IMG_PATH), f"❌ Target image not found: {TARGET_IMG_PATH}"
bgr_t = cv2.imread(TARGET_IMG_PATH)
rgb_t = cv2.cvtColor(bgr_t, cv2.COLOR_BGR2RGB)
lab_t = cv2.cvtColor(rgb_t, cv2.COLOR_RGB2LAB).astype(np.float32)
mask_t = tissue_mask_otsu_L(rgb_t)

target_stats = []
for ch in [lab_t[:, :, 0], lab_t[:, :, 1], lab_t[:, :, 2]]:
    m, s = robust_mean_std(ch, mask_t)
    assert m is not None, "❌ Target mask has too little tissue. Pick another target image."
    target_stats.append((m, s))

print("✅ Target stats (L,A,B):", target_stats)

# =========================
# MAIN
# =========================
for split in SPLITS:
    print(f"\n=== Split: {split} ===")

    in_dir  = os.path.join(INPUT_DIR, split, IMG_SUBDIR)
    out_dir = os.path.join(OUTPUT_DIR, split, IMG_SUBDIR)
    os.makedirs(out_dir, exist_ok=True)

    if os.path.exists(in_dir):
        files = [f for f in os.listdir(in_dir) if is_image_file(f)]
        print(f"Processing {split}/{IMG_SUBDIR}: {len(files)} images")

        for fname in tqdm(files, ncols=80):
            in_path  = os.path.join(in_dir, fname)
            out_path = os.path.join(out_dir, fname)

            bgr = cv2.imread(in_path)
            if bgr is None:
                continue
            rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

            rgb_norm = reinhard_normalize(rgb, target_stats)
            cv2.imwrite(out_path, cv2.cvtColor(rgb_norm, cv2.COLOR_RGB2BGR))
    else:
        print(f"⚠️ Missing folder (skip): {in_dir}")

    # Copy masks as-is
    for msub in MASK_SUBDIRS_TO_COPY:
        src = os.path.join(INPUT_DIR, split, msub)
        dst = os.path.join(OUTPUT_DIR, split, msub)
        if os.path.exists(src):
            print(f"Copying {split}/{msub} ...")
            copy_folder_if_exists(src, dst)

print("\n✅ DONE. New dataset created at:")
print(OUTPUT_DIR)

6. DATASET_liver_stu_MACENKO_SOFT (ABL-5)

In [ ]:
# =========================
# CONFIG
# =========================
INPUT_DIR  = "/content/drive/MyDrive/Challenge_Liver_MIP/DATASET_liver_stu"
OUTPUT_DIR = "/content/drive/MyDrive/Challenge_Liver_MIP/DATASET_liver_stu_MACENKO_SOFT"

SPLITS = ["train", "val", "test"]
IMG_SUBDIR = "image"
MASK_SUBDIRS_TO_COPY = ["manual", "manual_py"]

# Target image (representative, tissue-rich)
TARGET_IMG_PATH = "/content/drive/MyDrive/Challenge_Liver_MIP/DATASET_liver_stu/train/image/1001340_72.png"

# Softness: 0 -> original, 1 -> full Macenko
ALPHA_BLEND = 0.35  # ✅ start here

# Macenko parameters (safe)
OD_EPS = 1e-6
BETA = 0.15          # tissue threshold in OD space
ALPHA = 1.0          # percentile for stain vectors (usually 1)
OD_CLIP = 2.5        # clip OD to avoid blow-ups
C_CLIP = 5.0         # clip concentrations
BACKGROUND_V = 245    # for quick background detection

# =========================
# HELPERS
# =========================
def is_image_file(fname):
    f = fname.lower()
    return f.endswith((".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp"))

def rgb2od(I):
    I = I.astype(np.float32)
    I = np.clip(I, 1.0, 255.0)
    return -np.log((I + 1.0) / 256.0)

def od2rgb(OD):
    I = (np.exp(-OD) * 256.0) - 1.0
    return np.clip(I, 0, 255).astype(np.uint8)

def tissue_mask(img_rgb):
    # simple + safe: remove near-white background
    hsv = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2HSV)
    v = hsv[:, :, 2]
    mask = (v < BACKGROUND_V).astype(np.uint8)

    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, k, iterations=1)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, k, iterations=2)
    return mask

def get_stain_matrix_macenko(img_rgb):
    """
    Returns 3x2 stain matrix W (columns are stain vectors).
    """
    OD = rgb2od(img_rgb)
    OD = np.clip(OD, 0, OD_CLIP)

    m = tissue_mask(img_rgb)
    OD_tissue = OD[m == 1].reshape(-1, 3)

    # filter transparent pixels (Macenko beta)
    keep = np.linalg.norm(OD_tissue, axis=1) > BETA
    OD_tissue = OD_tissue[keep]
    if OD_tissue.shape[0] < 50:
        return None

    # PCA on OD
    cov = np.cov(OD_tissue.T)
    eigvals, eigvecs = np.linalg.eigh(cov)
    V = eigvecs[:, np.argsort(eigvals)[::-1]]  # descending

    # project on first 2 PCs
    proj = OD_tissue @ V[:, :2]
    angles = np.arctan2(proj[:, 1], proj[:, 0])

    # extreme angles
    min_a = np.percentile(angles, ALPHA)
    max_a = np.percentile(angles, 100 - ALPHA)

    v1 = V[:, :2] @ np.array([np.cos(min_a), np.sin(min_a)])
    v2 = V[:, :2] @ np.array([np.cos(max_a), np.sin(max_a)])

    # order stains consistently (H tends to have larger 1st component)
    if v1[0] < v2[0]:
        W = np.stack([v2, v1], axis=1)
    else:
        W = np.stack([v1, v2], axis=1)

    # normalize columns
    W = W / (np.linalg.norm(W, axis=0, keepdims=True) + 1e-8)
    return W

def get_concentrations(OD, W):
    # Solve OD = C * W^T  -> C = OD * pinv(W^T)
    # OD: (N,3), W: (3,2)
    pinv = np.linalg.pinv(W)
    C = OD @ pinv.T  # (N,2)
    C = np.clip(C, 0, C_CLIP)
    return C

def macenko_soft_normalize(img_rgb, W_src, W_tgt, Ctgt_ref):
    """
    Soft Macenko:
    - estimate concentrations in source
    - scale to target reference concentration stats
    - reconstruct using target stain matrix
    - blend with original
    """
    OD = rgb2od(img_rgb)
    OD = np.clip(OD, 0, OD_CLIP)

    H, W = img_rgb.shape[:2]
    OD_flat = OD.reshape(-1, 3)

    C_src = get_concentrations(OD_flat, W_src)

    # match concentration percentiles to target (robust scaling)
    # use 99th percentile for scaling
    p = 99
    src_p = np.percentile(C_src, p, axis=0) + 1e-6
    tgt_p = Ctgt_ref + 1e-6
    scale = tgt_p / src_p

    C_adj = C_src * scale[None, :]
    C_adj = np.clip(C_adj, 0, C_CLIP)

    OD_rec = (C_adj @ W_tgt.T).reshape(H, W, 3)
    rgb_rec = od2rgb(OD_rec)

    # Blend (soft)
    out = (1 - ALPHA_BLEND) * img_rgb.astype(np.float32) + ALPHA_BLEND * rgb_rec.astype(np.float32)
    return np.clip(out, 0, 255).astype(np.uint8)

def copy_folder_if_exists(src, dst):
    if os.path.exists(src):
        os.makedirs(dst, exist_ok=True)
        for fname in os.listdir(src):
            in_path = os.path.join(src, fname)
            out_path = os.path.join(dst, fname)
            if os.path.isfile(in_path):
                shutil.copy2(in_path, out_path)

# =========================
# PREPARE TARGET
# =========================
assert os.path.exists(TARGET_IMG_PATH), f"❌ Target image not found: {TARGET_IMG_PATH}"
bgr = cv2.imread(TARGET_IMG_PATH)
rgb_tgt = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

W_tgt = get_stain_matrix_macenko(rgb_tgt)
assert W_tgt is not None, "❌ Could not estimate stain matrix for target. Pick another target."

# reference concentration scale for target (99th percentile)
OD_tgt = np.clip(rgb2od(rgb_tgt), 0, OD_CLIP).reshape(-1, 3)
C_tgt = get_concentrations(OD_tgt, W_tgt)
Ctgt_ref = np.percentile(C_tgt, 99, axis=0)

print("✅ Target stain matrix W_tgt:\n", W_tgt)
print("✅ Target concentration ref (p99):", Ctgt_ref)
print("✅ ALPHA_BLEND (softness):", ALPHA_BLEND)

# =========================
# MAIN
# =========================
for split in SPLITS:
    print(f"\n=== Split: {split} ===")
    in_dir  = os.path.join(INPUT_DIR, split, IMG_SUBDIR)
    out_dir = os.path.join(OUTPUT_DIR, split, IMG_SUBDIR)
    os.makedirs(out_dir, exist_ok=True)

    if os.path.exists(in_dir):
        files = [f for f in os.listdir(in_dir) if is_image_file(f)]
        print(f"Processing {split}/{IMG_SUBDIR}: {len(files)} images")

        for fname in tqdm(files, ncols=80):
            in_path  = os.path.join(in_dir, fname)
            out_path = os.path.join(out_dir, fname)

            bgr = cv2.imread(in_path)
            if bgr is None:
                continue
            rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

            W_src = get_stain_matrix_macenko(rgb)
            if W_src is None:
                # if fails, save original (safe fallback)
                rgb_out = rgb
            else:
                rgb_out = macenko_soft_normalize(rgb, W_src, W_tgt, Ctgt_ref)

            cv2.imwrite(out_path, cv2.cvtColor(rgb_out, cv2.COLOR_RGB2BGR))
    else:
        print(f"⚠️ Missing folder (skip): {in_dir}")

    # Copy masks as-is
    for msub in MASK_SUBDIRS_TO_COPY:
        src = os.path.join(INPUT_DIR, split, msub)
        dst = os.path.join(OUTPUT_DIR, split, msub)
        if os.path.exists(src):
            print(f"Copying {split}/{msub} ...")
            copy_folder_if_exists(src, dst)

print("\n✅ DONE. New dataset created at:")
print(OUTPUT_DIR)

7. DATASET_liver_stu_MACENKO_SOFT_ABL_6

In [ ]:
Here is the clean English version with consistent paths and only Macenko soft normalization for the professor test set, copying manual and manual_py into the same output structure.

# =========================
# FINAL TEST ONLY — MACENKO SOFT + COPY manual/manual_py
#
# Final structure:
# FINAL_MODEL_DIR/
#   └── test_preprocessed/
#       ├── image/        (processed test images)
#       ├── manual/       (copied if exists)
#       └── manual_py/    (copied if exists)
# =========================

import os
import shutil
import cv2
import numpy as np
from tqdm import tqdm

# =========================
# CONFIG (CONSISTENT PATHS)
# =========================

# Input: professor test set (from your setup block)
INPUT_TEST_DIR = test_images_dir

# Base dataset directory (used only to copy manual/manual_py if present)
INPUT_DIR = "/content/drive/MyDrive/Challenge_Liver_MIP/DATASET_liver_stu"

# Output base: inside the EXISTING model folder
OUTPUT_BASE_DIR = os.path.join(FINAL_MODEL_DIR, "test_preprocessed")
OUTPUT_TEST_DIR = os.path.join(OUTPUT_BASE_DIR, "image")
OUTPUT_MANUAL_DIR = os.path.join(OUTPUT_BASE_DIR, "manual")
OUTPUT_MANUAL_PY_DIR = os.path.join(OUTPUT_BASE_DIR, "manual_py")

os.makedirs(OUTPUT_TEST_DIR, exist_ok=True)
os.makedirs(OUTPUT_MANUAL_DIR, exist_ok=True)
os.makedirs(OUTPUT_MANUAL_PY_DIR, exist_ok=True)

# Optional redirect for inference
test_images_dir_pre = OUTPUT_TEST_DIR

# Target image (representative, tissue-rich)
TARGET_IMG_PATH = os.path.join(INPUT_DIR, "train", "image", "1001340_72.png")

# If manual/manual_py exist in splits, copy them
SPLITS = ["train", "val", "test"]
SUBFOLDERS_TO_COPY = ["manual", "manual_py"]

# Soft blending factor (0 = original, 1 = full Macenko)
ALPHA_BLEND = 0.2

# Macenko parameters (stable settings)
OD_EPS = 1e-6
BETA = 0.15
ALPHA = 1.0
OD_CLIP = 2.5
C_CLIP = 5.0
BACKGROUND_V = 245

# =========================
# HELPERS
# =========================

def is_image_file(fname):
    return fname.lower().endswith((".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp"))

def rgb2od(I):
    I = I.astype(np.float32)
    I = np.clip(I, 1.0, 255.0)
    return -np.log((I + 1.0) / 256.0)

def od2rgb(OD):
    I = (np.exp(-OD) * 256.0) - 1.0
    return np.clip(I, 0, 255).astype(np.uint8)

def tissue_mask(img_rgb):
    hsv = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2HSV)
    v = hsv[:, :, 2]
    mask = (v < BACKGROUND_V).astype(np.uint8)

    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel, iterations=1)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=2)
    return mask

def get_stain_matrix_macenko(img_rgb):
    OD = rgb2od(img_rgb)
    OD = np.clip(OD, 0, OD_CLIP)

    mask = tissue_mask(img_rgb)
    OD_tissue = OD[mask == 1].reshape(-1, 3)

    keep = np.linalg.norm(OD_tissue, axis=1) > BETA
    OD_tissue = OD_tissue[keep]
    if OD_tissue.shape[0] < 50:
        return None

    cov = np.cov(OD_tissue.T)
    eigvals, eigvecs = np.linalg.eigh(cov)
    V = eigvecs[:, np.argsort(eigvals)[::-1]]

    proj = OD_tissue @ V[:, :2]
    angles = np.arctan2(proj[:, 1], proj[:, 0])

    min_a = np.percentile(angles, ALPHA)
    max_a = np.percentile(angles, 100 - ALPHA)

    v1 = V[:, :2] @ np.array([np.cos(min_a), np.sin(min_a)])
    v2 = V[:, :2] @ np.array([np.cos(max_a), np.sin(max_a)])

    if v1[0] < v2[0]:
        W = np.stack([v2, v1], axis=1)
    else:
        W = np.stack([v1, v2], axis=1)

    W = W / (np.linalg.norm(W, axis=0, keepdims=True) + 1e-8)
    return W

def get_concentrations(OD, W):
    pinv = np.linalg.pinv(W)
    C = OD @ pinv.T
    return np.clip(C, 0, C_CLIP)

def macenko_soft_normalize(img_rgb, W_src, W_tgt, Ctgt_ref):
    OD = rgb2od(img_rgb)
    OD = np.clip(OD, 0, OD_CLIP)

    H, W = img_rgb.shape[:2]
    OD_flat = OD.reshape(-1, 3)

    C_src = get_concentrations(OD_flat, W_src)

    src_p = np.percentile(C_src, 99, axis=0) + 1e-6
    tgt_p = Ctgt_ref + 1e-6
    scale = tgt_p / src_p

    C_adj = np.clip(C_src * scale[None, :], 0, C_CLIP)

    OD_rec = (C_adj @ W_tgt.T).reshape(H, W, 3)
    rgb_rec = od2rgb(OD_rec)

    blended = (1 - ALPHA_BLEND) * img_rgb.astype(np.float32) + \
              ALPHA_BLEND * rgb_rec.astype(np.float32)

    return np.clip(blended, 0, 255).astype(np.uint8)

def copy_folder_if_exists(src, dst):
    if os.path.exists(src):
        os.makedirs(dst, exist_ok=True)
        for fname in os.listdir(src):
            src_path = os.path.join(src, fname)
            dst_path = os.path.join(dst, fname)
            if os.path.isfile(src_path):
                shutil.copy2(src_path, dst_path)

# =========================
# PREPARE TARGET
# =========================

assert os.path.exists(TARGET_IMG_PATH), f"Target image not found: {TARGET_IMG_PATH}"
bgr = cv2.imread(TARGET_IMG_PATH)
assert bgr is not None, f"Could not read target image: {TARGET_IMG_PATH}"
rgb_tgt = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

W_tgt = get_stain_matrix_macenko(rgb_tgt)
assert W_tgt is not None, "Could not estimate stain matrix for target."

OD_tgt = np.clip(rgb2od(rgb_tgt), 0, OD_CLIP).reshape(-1, 3)
C_tgt = get_concentrations(OD_tgt, W_tgt)
Ctgt_ref = np.percentile(C_tgt, 99, axis=0)

print("Target stain matrix estimated.")
print("Soft blending factor:", ALPHA_BLEND)

# =========================
# PROCESS PROFESSOR TEST SET
# =========================

assert os.path.exists(INPUT_TEST_DIR), f"Missing test directory: {INPUT_TEST_DIR}"
files = sorted([f for f in os.listdir(INPUT_TEST_DIR) if is_image_file(f)])
assert len(files) > 0, "No test images found."

print(f"\nProcessing {len(files)} test images...")

for fname in tqdm(files, ncols=80):
    in_path = os.path.join(INPUT_TEST_DIR, fname)
    out_path = os.path.join(OUTPUT_TEST_DIR, fname)

    bgr = cv2.imread(in_path)
    if bgr is None:
        continue

    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

    W_src = get_stain_matrix_macenko(rgb)
    if W_src is None:
        rgb_out = rgb  # fallback
    else:
        rgb_out = macenko_soft_normalize(rgb, W_src, W_tgt, Ctgt_ref)

    cv2.imwrite(out_path, cv2.cvtColor(rgb_out, cv2.COLOR_RGB2BGR))

print("\nTest preprocessing completed.")

# =========================
# COPY manual / manual_py IF PRESENT
# =========================

for split in SPLITS:
    for sub in SUBFOLDERS_TO_COPY:
        src = os.path.join(INPUT_DIR, split, sub)
        dst = OUTPUT_MANUAL_DIR if sub == "manual" else OUTPUT_MANUAL_PY_DIR

        if os.path.exists(src):
            print(f"Copying {src} → {dst}")
            copy_folder_if_exists(src, dst)

print("\nFinal folder created at:")
print(OUTPUT_BASE_DIR)

# Redirect for inference
test_images_dir = OUTPUT_TEST_DIR
print("test_images_dir redirected to:", test_images_dir)

8. DATASET_liver_stu_MACENKO_MT_SOFT_ABL_7

In [ ]:
# =========================
# CONFIG
# =========================
INPUT_DIR  = "/content/drive/MyDrive/Challenge_Liver_MIP/DATASET_liver_stu"
OUTPUT_DIR = "/content/drive/MyDrive/Challenge_Liver_MIP/DATASET_liver_stu_MACENKO_MT_SOFT_ABL_7"

SPLITS = ["train", "val", "test"]
IMG_SUBDIR = "image"
MASK_SUBDIRS_TO_COPY = ["manual", "manual_py"]

# 👇 Put 3 representative target images here (tissue-rich, normal staining, clean vacuoles)
TARGET_IMG_PATHS = [
    "/content/drive/MyDrive/Challenge_Liver_MIP/DATASET_liver_stu/train/image/1001340_72.png",
    "/content/drive/MyDrive/Challenge_Liver_MIP/DATASET_liver_stu/train/image/1004135_49.png",
    "/content/drive/MyDrive/Challenge_Liver_MIP/DATASET_liver_stu/train/image/1004062_75.png",
]

# Softness: 0 -> original, 1 -> full Macenko
ALPHA_BLEND = 0.20  # ✅ best you found

# Macenko parameters (safe)
OD_EPS = 1e-6
BETA = 0.15          # tissue threshold in OD space
ALPHA = 1.0          # percentile for stain vectors
OD_CLIP = 2.5        # clip OD to avoid blow-ups
C_CLIP = 5.0         # clip concentrations
BACKGROUND_V = 245    # background cutoff for tissue mask
MIN_TISSUE_FRAC = 0.10  # if <10% tissue -> keep original

# Reproducibility (optional)
RANDOM_SEED = 123
random.seed(RANDOM_SEED)

# =========================
# HELPERS
# =========================
def is_image_file(fname):
    f = fname.lower()
    return f.endswith((".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp"))

def rgb2od(I):
    I = I.astype(np.float32)
    I = np.clip(I, 1.0, 255.0)
    return -np.log((I + 1.0) / 256.0)

def od2rgb(OD):
    I = (np.exp(-OD) * 256.0) - 1.0
    return np.clip(I, 0, 255).astype(np.uint8)

def tissue_mask(img_rgb):
    hsv = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2HSV)
    v = hsv[:, :, 2]
    mask = (v < BACKGROUND_V).astype(np.uint8)

    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, k, iterations=1)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, k, iterations=2)
    return mask

def get_stain_matrix_macenko(img_rgb):
    OD = rgb2od(img_rgb)
    OD = np.clip(OD, 0, OD_CLIP)

    m = tissue_mask(img_rgb)
    tissue_frac = float(m.mean())
    if tissue_frac < MIN_TISSUE_FRAC:
        return None

    OD_tissue = OD[m == 1].reshape(-1, 3)

    keep = np.linalg.norm(OD_tissue, axis=1) > BETA
    OD_tissue = OD_tissue[keep]
    if OD_tissue.shape[0] < 50:
        return None

    cov = np.cov(OD_tissue.T)
    eigvals, eigvecs = np.linalg.eigh(cov)
    V = eigvecs[:, np.argsort(eigvals)[::-1]]

    proj = OD_tissue @ V[:, :2]
    angles = np.arctan2(proj[:, 1], proj[:, 0])

    min_a = np.percentile(angles, ALPHA)
    max_a = np.percentile(angles, 100 - ALPHA)

    v1 = V[:, :2] @ np.array([np.cos(min_a), np.sin(min_a)])
    v2 = V[:, :2] @ np.array([np.cos(max_a), np.sin(max_a)])

    # consistent ordering
    if v1[0] < v2[0]:
        W = np.stack([v2, v1], axis=1)
    else:
        W = np.stack([v1, v2], axis=1)

    W = W / (np.linalg.norm(W, axis=0, keepdims=True) + 1e-8)
    return W

def get_concentrations(OD_flat, W):
    pinv = np.linalg.pinv(W)
    C = OD_flat @ pinv.T
    C = np.clip(C, 0, C_CLIP)
    return C

def macenko_soft_normalize(img_rgb, W_src, W_tgt, Ctgt_ref):
    OD = rgb2od(img_rgb)
    OD = np.clip(OD, 0, OD_CLIP)

    H, W = img_rgb.shape[:2]
    OD_flat = OD.reshape(-1, 3)

    C_src = get_concentrations(OD_flat, W_src)

    # robust scale using 99th percentile
    src_p = np.percentile(C_src, 99, axis=0) + 1e-6
    tgt_p = Ctgt_ref + 1e-6
    scale = tgt_p / src_p

    C_adj = np.clip(C_src * scale[None, :], 0, C_CLIP)
    OD_rec = (C_adj @ W_tgt.T).reshape(H, W, 3)
    rgb_rec = od2rgb(OD_rec)

    # soft blend
    out = (1 - ALPHA_BLEND) * img_rgb.astype(np.float32) + ALPHA_BLEND * rgb_rec.astype(np.float32)
    return np.clip(out, 0, 255).astype(np.uint8)

def copy_folder_if_exists(src, dst):
    if os.path.exists(src):
        os.makedirs(dst, exist_ok=True)
        for fname in os.listdir(src):
            in_path = os.path.join(src, fname)
            out_path = os.path.join(dst, fname)
            if os.path.isfile(in_path):
                shutil.copy2(in_path, out_path)

# =========================
# PREPARE TARGETS (precompute W and Ctgt_ref for each target)
# =========================
targets = []
for p in TARGET_IMG_PATHS:
    assert os.path.exists(p), f"❌ Target not found: {p}"
    bgr = cv2.imread(p)
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

    W_tgt = get_stain_matrix_macenko(rgb)
    assert W_tgt is not None, f"❌ Failed stain matrix for target: {p} (choose another)"

    OD_tgt = np.clip(rgb2od(rgb), 0, OD_CLIP).reshape(-1, 3)
    C_tgt = get_concentrations(OD_tgt, W_tgt)
    Ctgt_ref = np.percentile(C_tgt, 99, axis=0)

    targets.append((p, W_tgt, Ctgt_ref))

print("✅ Prepared", len(targets), "targets")
print("✅ ALPHA_BLEND =", ALPHA_BLEND)

# =========================
# MAIN
# =========================
for split in SPLITS:
    print(f"\n=== Split: {split} ===")
    in_dir  = os.path.join(INPUT_DIR, split, IMG_SUBDIR)
    out_dir = os.path.join(OUTPUT_DIR, split, IMG_SUBDIR)
    os.makedirs(out_dir, exist_ok=True)

    if os.path.exists(in_dir):
        files = [f for f in os.listdir(in_dir) if is_image_file(f)]
        print(f"Processing {split}/{IMG_SUBDIR}: {len(files)} images")

        for fname in tqdm(files, ncols=80):
            in_path  = os.path.join(in_dir, fname)
            out_path = os.path.join(out_dir, fname)

            bgr = cv2.imread(in_path)
            if bgr is None:
                continue
            rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

            W_src = get_stain_matrix_macenko(rgb)
            if W_src is None:
                rgb_out = rgb  # safe fallback
            else:
                # pick one of the 3 targets at random
                _, W_tgt, Ctgt_ref = random.choice(targets)
                rgb_out = macenko_soft_normalize(rgb, W_src, W_tgt, Ctgt_ref)

            cv2.imwrite(out_path, cv2.cvtColor(rgb_out, cv2.COLOR_RGB2BGR))
    else:
        print(f"⚠️ Missing folder (skip): {in_dir}")

    # Copy masks as-is
    for msub in MASK_SUBDIRS_TO_COPY:
        src = os.path.join(INPUT_DIR, split, msub)
        dst = os.path.join(OUTPUT_DIR, split, msub)
        if os.path.exists(src):
            print(f"Copying {split}/{msub} ...")
            copy_folder_if_exists(src, dst)

print("\n✅ DONE. New dataset created at:")
print(OUTPUT_DIR)

9. DATASET_liver_stu_VAHADANE_SOFT_ABL

In [ ]:
# =========================
# CONFIG
# =========================
INPUT_DIR  = "/content/drive/MyDrive/Challenge_Liver_MIP/DATASET_liver_stu"
OUTPUT_DIR = "/content/drive/MyDrive/Challenge_Liver_MIP/DATASET_liver_stu_VAHADANE_SOFT_ABL"

SPLITS = ["train", "val", "test"]
IMG_SUBDIR = "image"
MASK_SUBDIRS_TO_COPY = ["manual", "manual_py"]

# Target image (representative, tissue-rich)
TARGET_IMG_PATH = "/content/drive/MyDrive/Challenge_Liver_MIP/DATASET_liver_stu/train/image/1001340_72.png"

# Softness: 0 -> original, 1 -> full normalization
ALPHA_BLEND = 0.2  # ✅ start here

# OD / masking params
BETA = 0.15          # OD threshold to ignore near-transparent pixels
OD_CLIP = 2.5
C_CLIP = 5.0
BACKGROUND_V = 245

# Vahadane / NMF params
MAX_TISSUE_PIXELS = 200000  # sample for speed/robustness
NMF_MAX_ITER = 400
NMF_RANDOM_STATE = 0

NMF_ALPHA_W = 0.02  # sparsity on concentrations
NMF_ALPHA_H = 0.02  # sparsity on stain vectors
NMF_L1_RATIO = 1.0  # 1.0 => L1, 0.0 => L2

# NNLS-ish concentration estimation
NNLS_ITERS = 50
EPS = 1e-8

# =========================
# HELPERS
# =========================
def is_image_file(fname):
    f = fname.lower()
    return f.endswith((".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp"))

def rgb2od(I):
    I = I.astype(np.float32)
    I = np.clip(I, 1.0, 255.0)
    return -np.log((I + 1.0) / 256.0)

def od2rgb(OD):
    I = (np.exp(-OD) * 256.0) - 1.0
    return np.clip(I, 0, 255).astype(np.uint8)

def tissue_mask(img_rgb):
    hsv = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2HSV)
    v = hsv[:, :, 2]
    mask = (v < BACKGROUND_V).astype(np.uint8)

    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, k, iterations=1)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, k, iterations=2)
    return mask

def copy_folder_if_exists(src, dst):
    if os.path.exists(src):
        os.makedirs(dst, exist_ok=True)
        for fname in os.listdir(src):
            in_path = os.path.join(src, fname)
            out_path = os.path.join(dst, fname)
            if os.path.isfile(in_path):
                shutil.copy2(in_path, out_path)

# =========================
# VAHADANE CORE
# =========================
def _sample_rows(X, max_rows, seed=0):
    if X.shape[0] <= max_rows:
        return X
    rng = np.random.default_rng(seed)
    idx = rng.choice(X.shape[0], size=max_rows, replace=False)
    return X[idx]

def get_stain_matrix_vahadane(img_rgb):
    """
    Estimates 3x2 stain matrix using Vahadane-like method:
      - OD in tissue
      - NMF (non-negative) with L1 regularization
    Returns W (3,2) with normalized columns.
    """
    try:
        from sklearn.decomposition import NMF
    except Exception as e:
        raise ImportError("scikit-learn is missing. Install with: pip install scikit-learn") from e

    OD = rgb2od(img_rgb)
    OD = np.clip(OD, 0, OD_CLIP)

    m = tissue_mask(img_rgb)
    OD_tissue = OD[m == 1].reshape(-1, 3)

    # filter highly transparent pixels
    keep = np.linalg.norm(OD_tissue, axis=1) > BETA
    OD_tissue = OD_tissue[keep]

    if OD_tissue.shape[0] < 200:
        return None

    # sample for speed
    OD_tissue = _sample_rows(OD_tissue, MAX_TISSUE_PIXELS, seed=NMF_RANDOM_STATE)

    # NMF expects non-negative matrix (ok: OD>=0)
    nmf = NMF(
        n_components=2,
        init="nndsvda",
        solver="cd",
        beta_loss="frobenius",
        max_iter=NMF_MAX_ITER,
        random_state=NMF_RANDOM_STATE,
        alpha_W=NMF_ALPHA_W,
        alpha_H=NMF_ALPHA_H,
        l1_ratio=NMF_L1_RATIO
    )

    # X ≈ W_nmf @ H_nmf
    W_nmf = nmf.fit_transform(OD_tissue)  # (N,2) concentrations (up to scaling)
    H_nmf = nmf.components_               # (2,3)

    W = H_nmf.T  # (3,2) stain vectors
    # normalize columns
    W = W / (np.linalg.norm(W, axis=0, keepdims=True) + 1e-8)

    # consistent ordering (heuristic: H usually "peaks" more in R)
    if W[0, 0] < W[0, 1]:
        W = W[:, [1, 0]]

    return W

def get_concentrations_nnls_mult(OD_flat, W_stain, iters=NNLS_ITERS):
    """
    Estimates C (N,2) >=0 for OD ≈ C @ W^T using multiplicative updates (NMF-like).
    OD_flat: (N,3), W_stain: (3,2)
    """
    # initialization: lstsq + clip (fast)
    pinv = np.linalg.pinv(W_stain)
    C = OD_flat @ pinv.T  # (N,2)
    C = np.clip(C, 0, C_CLIP).astype(np.float32)

    WT = W_stain.T.astype(np.float32)            # (2,3)
    WTW = WT @ W_stain.astype(np.float32)        # (2,2)
    WTX = WT @ OD_flat.T.astype(np.float32)      # (2,N)

    # updates: C <- C * (WTX / (WTW*C^T + eps))
    Ct = C.T  # (2,N)
    for _ in range(iters):
        denom = (WTW @ Ct) + EPS
        Ct *= (WTX / denom)
    C = Ct.T
    C = np.clip(C, 0, C_CLIP)
    return C

def vahadane_soft_normalize(img_rgb, W_src, W_tgt, Ctgt_ref):
    """
    Soft Vahadane:
      - Concentrations via NNLS-ish
      - Robust scaling using target p99
      - Reconstruct using W_tgt
      - Blend with original
    Preserves background using the tissue mask.
    """
    OD = rgb2od(img_rgb)
    OD = np.clip(OD, 0, OD_CLIP)

    H, W = img_rgb.shape[:2]
    OD_flat = OD.reshape(-1, 3)

    # mask to preserve background
    m = tissue_mask(img_rgb).reshape(-1).astype(bool)

    # concentrations ONLY in tissue (more stable)
    C_src = np.zeros((OD_flat.shape[0], 2), dtype=np.float32)
    if m.sum() < 50:
        return img_rgb

    C_tissue = get_concentrations_nnls_mult(OD_flat[m], W_src, iters=NNLS_ITERS)
    C_src[m] = C_tissue

    # robust scaling (p99) using only tissue
    p = 99
    src_p = np.percentile(C_tissue, p, axis=0) + 1e-6
    tgt_p = Ctgt_ref + 1e-6
    scale = tgt_p / src_p

    C_adj = C_src * scale[None, :]
    C_adj = np.clip(C_adj, 0, C_CLIP)

    OD_rec = (C_adj @ W_tgt.T).reshape(H, W, 3)
    rgb_rec = od2rgb(OD_rec)

    # soft blend + preserve original background outside tissue
    out = (1 - ALPHA_BLEND) * img_rgb.astype(np.float32) + ALPHA_BLEND * rgb_rec.astype(np.float32)
    out = np.clip(out, 0, 255).astype(np.uint8)

    # explicitly preserve background
    out.reshape(-1, 3)[~m] = img_rgb.reshape(-1, 3)[~m]
    return out

# =========================
# PREPARE TARGET
# =========================
assert os.path.exists(TARGET_IMG_PATH), f"❌ Target image not found: {TARGET_IMG_PATH}"
bgr = cv2.imread(TARGET_IMG_PATH)
rgb_tgt = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

W_tgt = get_stain_matrix_vahadane(rgb_tgt)
assert W_tgt is not None, "❌ Could not estimate stain matrix for target. Pick another target."

# reference concentration scale for target (99th percentile) – ONLY tissue
OD_tgt = np.clip(rgb2od(rgb_tgt), 0, OD_CLIP).reshape(-1, 3)
m_tgt = tissue_mask(rgb_tgt).reshape(-1).astype(bool)
C_tgt = get_concentrations_nnls_mult(OD_tgt[m_tgt], W_tgt, iters=NNLS_ITERS)
Ctgt_ref = np.percentile(C_tgt, 99, axis=0)

print("✅ Target stain matrix W_tgt (Vahadane):\n", W_tgt)
print("✅ Target concentration ref (p99):", Ctgt_ref)
print("✅ ALPHA_BLEND (softness):", ALPHA_BLEND)

# =========================
# MAIN
# =========================
for split in SPLITS:
    print(f"\n=== Split: {split} ===")
    in_dir  = os.path.join(INPUT_DIR, split, IMG_SUBDIR)
    out_dir = os.path.join(OUTPUT_DIR, split, IMG_SUBDIR)
    os.makedirs(out_dir, exist_ok=True)

    if os.path.exists(in_dir):
        files = [f for f in os.listdir(in_dir) if is_image_file(f)]
        print(f"Processing {split}/{IMG_SUBDIR}: {len(files)} images")

        for fname in tqdm(files, ncols=80):
            in_path  = os.path.join(in_dir, fname)
            out_path = os.path.join(out_dir, fname)

            bgr = cv2.imread(in_path)
            if bgr is None:
                continue
            rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

            W_src = get_stain_matrix_vahadane(rgb)
            if W_src is None:
                rgb_out = rgb  # fallback
            else:
                rgb_out = vahadane_soft_normalize(rgb, W_src, W_tgt, Ctgt_ref)

            cv2.imwrite(out_path, cv2.cvtColor(rgb_out, cv2.COLOR_RGB2BGR))
    else:
        print(f"⚠️ Missing folder (skip): {in_dir}")

    # Copy masks as-is
    for msub in MASK_SUBDIRS_TO_COPY:
        src = os.path.join(INPUT_DIR, split, msub)
        dst = os.path.join(OUTPUT_DIR, split, msub)
        if os.path.exists(src):
            print(f"Copying {split}/{msub} ...")
            copy_folder_if_exists(src, dst)

print("\n✅ DONE. New dataset created at:")
print(OUTPUT_DIR)
